# dbt Analytics Engineering — Crash Course> **Engineering Crash Courses** · [Web verzió](./index.html) · [Vissza a főoldalra](../index.html)Ez egy futtatható **Jupyter notebook** formátum, párhuzamosan a web-alapú kurzussal.Itt ugyanazokat a kódrészleteket tudod lokálisan, saját környezetben végigcsinálni.## Hogyan futtasd```bash# 1. Virtuális környezet (Python 3.10+)python -m venv .venv# Windows:.venv\Scripts\activate# macOS/Linux:source .venv/bin/activate# 2. Telepítsd a függőségeket (a notebook első cellája)# 3. Indítsd a Jupytertjupyter lab# vagyjupyter notebook```Minden cella saját magában értelmezhető. A `# %%` kommentek Jupytekben és VSCode-ban is a cellák határát jelölik.

## 1. Környezet — dbt-duckdb (nem kell warehouse)

In [ ]:
%pip install dbt-duckdb --quiet

In [ ]:
!dbt --version

## 2. dbt projekt létrehozása

In [ ]:
import osfrom pathlib import Pathproject_root = Path('webshop_dbt')project_root.mkdir(exist_ok=True)# dbt_project.yml(project_root / 'dbt_project.yml').write_text('''name: webshopversion: "1.0.0"config-version: 2profile: webshopmodel-paths: ["models"]target-path: "target"clean-targets: ["target", "dbt_packages"]models:  webshop:    staging:       { +materialized: view }    intermediate:  { +materialized: view }    marts:         { +materialized: table }''')# profiles.yml a gyökérben (lokális DuckDB)(project_root / 'profiles.yml').write_text(f'''webshop:  target: dev  outputs:    dev:      type: duckdb      path: {(project_root / "webshop.duckdb").absolute()}      threads: 4''')print('Projekt létrehozva:', project_root.absolute())

## 3. Seed (CSV → tábla) és modellek

In [ ]:
(project_root / 'seeds').mkdir(exist_ok=True)(project_root / 'seeds' / 'raw_orders.csv').write_text('''order_id,customer_id,amount,status,order_date1,1,12500,paid,2025-01-202,1,8900,paid,2025-02-153,2,24000,pending,2025-03-014,3,5200,paid,2025-03-155,1,3400,cancelled,2025-04-10''')# Staging modell(project_root / 'models' / 'staging').mkdir(parents=True, exist_ok=True)(project_root / 'models' / 'staging' / 'stg_orders.sql').write_text('''SELECT    order_id,    customer_id,    CAST(amount AS DECIMAL(10,2)) AS amount,    LOWER(status)                 AS status,    CAST(order_date AS DATE)      AS order_dateFROM {{ ref('raw_orders') }}''')# Mart modell (agregált)(project_root / 'models' / 'marts').mkdir(exist_ok=True)(project_root / 'models' / 'marts' / 'mart_revenue_by_customer.sql').write_text('''SELECT    customer_id,    COUNT(*)    AS orders,    SUM(amount) AS revenueFROM {{ ref('stg_orders') }}WHERE status = 'paid'GROUP BY customer_idORDER BY revenue DESC''')# Tesztek(project_root / 'models' / 'staging' / 'schema.yml').write_text('''version: 2models:  - name: stg_orders    columns:      - name: order_id        tests: [unique, not_null]      - name: amount        tests: [not_null]''')print('Seed, modellek és tesztek kész')

## 4. dbt futtatás

In [ ]:
import subprocessenv = os.environ.copy()env['DBT_PROFILES_DIR'] = str(project_root.absolute())# Seed betöltés → modellek → tesztekfor cmd in ['seed', 'run', 'test']:    print(f'\n=== dbt {cmd} ===')    result = subprocess.run(['dbt', cmd, '--project-dir', str(project_root)],                            env=env, capture_output=True, text=True)    print(result.stdout[-800:] if result.stdout else '(no stdout)')    if result.returncode != 0:        print('STDERR:', result.stderr[-400:])

## 5. Eredmény lekérdezése

In [ ]:
import duckdbcon = duckdb.connect(str(project_root / 'webshop.duckdb'))print(con.execute('SHOW TABLES').df())print('\n--- mart_revenue_by_customer ---')print(con.execute('SELECT * FROM mart_revenue_by_customer').df())con.close()

## Következő lépések- Térj vissza a [web-alapú kurzushoz](dbt-analytics-engineering/index.html) a teljes anyagért, diagramokért és kvízekért.- Kapcsolódó források és videók a kurzusoldal alján találhatók a "További tanulás" szekcióban.- Ha elakadsz: [GitHub Issues](https://github.com/lugosidomotor/engineering_crash_courses/issues)---*Engineering Crash Courses · MIT licenc · Magyar Data & AI Engineering kurzusok*